In [ ]:

import sys
import os

sys.path.append(os.getcwd())

from projectroot import add_project_root

add_project_root()

In [ ]:
from databricks.vector_search.client import VectorSearchClient
from datetime import timedelta
import time
from configs.project import get_project_config

In [ ]:
projectConfig = get_project_config()
vs_config = projectConfig.vector_search_attributes["id_1"]

for k, v in vs_config.model_dump().items():
  print(k, v)

In [ ]:
vsc = VectorSearchClient(disable_notice=True)

In [ ]:
try:
    vsc.create_endpoint(name=vs_config.endpoint_name,
                        endpoint_type="STANDARD")
    
    time.sleep(5)

    vsc.wait_for_endpoint(name=vs_config.endpoint_name,
                                timeout=timedelta(minutes=60),
                                verbose=True)
    
    print(f"Endpoint named {vs_config.endpoint_name} is ready.")

    ep = vsc.get_endpoint(name=vs_config.endpoint_name)

except Exception as e:
    if "already exists" in str(e):
        print(f"Endpoint named {vs_config.endpoint_name} already exists.")
        ep = vsc.get_endpoint(name=vs_config.endpoint_name)
    else:
        raise e


In [ ]:
from databricks.sdk.service import iam
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.permissions.set(
    request_object_type="vector-search-endpoints",
    request_object_id=ep["id"],
    access_control_list=[
        iam.AccessControlRequest(
            group_name="users", permission_level=iam.PermissionLevel.CAN_MANAGE
        )
    ],
)